This notebook performs DESeq2 pseudobulk differential expression analysis from snRNA-seq data.

## Setup

In [1]:
# Set paths
SOD1_RENV_RNA="/oak/stanford/groups/agitler/Shared/Shared_Jupyter_Notebook_Analysis/4.1.1-OG-RNA/"
renv::load(SOD1_RENV_RNA)

In [2]:
# Load packages
suppressPackageStartupMessages({
  library(Seurat)
  library(SoupX)
  library(ggplot2)
  library(dplyr)
  library(BiocParallel)
  library(scDblFinder)
  library(svglite)
  library(pheatmap)
  library(matrixStats)
  library(tibble)
  library(DESeq2)
  library(VennDiagram)
  library(grid)
  library(stringr)
  library(ggrepel)
  library(ggrastr)
  library(ggpubr)
  library(forcats)
  library(ggbreak)
  library(cowplot)
  library(colorspace)
  library(broom)
  library(colorspace)
})

## Directory to save output

In [3]:
save_dir <- '/oak/stanford/groups/agitler/Shared/SOD1_Paper/RNA/'

## DESeq2 differential expression analysis functions

In [4]:
prepare_counts <- function(seurat_object, test.category, by.category, min.nuclei) {
    seurat_object$samples <- paste0(seurat_object@meta.data[, test.category], "_", seurat_object$orig.ident)
    
    # Aggregate counts for each sample
    counts <- AggregateExpression(seurat_object, 
                    group.by = c(by.category, "samples"),
                    assays = "RNA",
                    slot = "counts",
                    return.seurat = FALSE)
    
    counts <- counts$RNA
    
    # Transpose counts and convert to data frame
    counts.t.df <- as.data.frame(t(counts))
    
    # Get samples to delete (number of nuclei < min.nuclei)
    # 1. Get table containing nuclei counts for each sample/by.category pair
    nuclei_count_table <- table(seurat_object@meta.data[, by.category], seurat_object$samples)
    
    # 2. Get row names (by.category) and column names (samples)
    by_category <- rownames(nuclei_count_table)
    samples <- colnames(nuclei_count_table)

    # 3. Iterate over all combinations, check the condition, and store samples to delete in the vector
    to_delete <- c()

    for (i in by_category) {
      for (j in samples) {
        value <- nuclei_count_table[i, j]
        if (value < min.nuclei) {
          to_delete <- c(to_delete, paste(i, j, sep = "_"))
        }
      }
    }
    
    # 4. Filter out sample/by.category pairs with fewer than min.nuclei 
    counts.t.df <- counts.t.df %>% filter(!row.names(.) %in% to_delete)
    
    # Split data frame by by.category
    splitRows <- sub("_.*", "", rownames(counts.t.df))
    counts.split <- split.data.frame(counts.t.df, f = factor(splitRows)) 
    
    # For every element in counts.split, remove by.category from the row names and then transpose
    counts.split <- lapply(counts.split, function(x){
    rownames(x) <- sub("^[^_]*_", "", rownames(x))
    t(x)
    })
    
}

In [5]:
remove_conditions <- function(matrix_list) {
  # Function to extract prefixes from column names
  extract_prefixes <- function(col_names) {
    sapply(col_names, function(col_name) {
      substr(col_name, 1, regexpr("_", col_name) - 1)
    })
  }
  
  # Remove columns with prefixes that occur only once (only one replicate of a condition)
  cleaned_matrix_list <- lapply(matrix_list, function(matrix) {
    prefixes <- extract_prefixes(colnames(matrix))  
    occurrences <- table(prefixes)
    prefixes_to_remove <- names(occurrences)[occurrences == 1]
    
    columns_to_remove <- colnames(matrix)[sapply(colnames(matrix), function(col_name) {
      any(sapply(prefixes_to_remove, function(prefix) {
        startsWith(col_name, prefix)
      }))
    })]      
      
    matrix[, !(colnames(matrix) %in% columns_to_remove), drop = FALSE]          
  })
    
  # Function to check if all columns have the same prefix (only one condition present)
  all_columns_have_same_prefix <- function(matrix) {
    prefixes <- extract_prefixes(colnames(matrix))
    length(unique(prefixes)) <= 1
  }
  
  # Remove matrices with all columns having the same prefix
  cleaned_matrix_list <- cleaned_matrix_list[!sapply(cleaned_matrix_list, all_columns_have_same_prefix)]
  
  return(cleaned_matrix_list)
}

In [6]:
run_deseq2 <- function(category.name, counts, savedir.path) {
    # Create a data frame of sample information
    colData <- data.frame(samples = colnames(counts))
    # Add a condition column containing the test.category information (ex: stage) for each sample
    colData$condition <- sub("_.*", "", colData$samples) 
    # Convert the sample names to row names
    colData <- colData %>% column_to_rownames(var = "samples")
    
    # Create DESEq object
    dds <- DESeqDataSetFromMatrix(countData = counts,
                      colData = colData,
                      design = ~ condition)
    
    # Run DESeq
    dds <- DESeq(dds, minReplicatesForReplace = Inf)
    
    # Loop over differential expression results and save 
    results_names <- resultsNames(dds)
    results_names <- results_names[grep("^condition", results_names)]
    
    for (result_name in results_names) {
        # Create a data frame containing the differential expression results
        res <- results(dds, name = result_name) %>% as.data.frame(.) %>% arrange(padj) %>% filter(!is.na(padj))
        # Save the results
        write.csv(res, file=paste0(savedir.path, paste0(category.name, '_', sub("^[^_]*_", "", result_name), '.csv')))
    }
}

## Non-downsampled DESeq2 analyses

### Differential expression in cholinergic types with disease (all data)
Cholinergic_Type

In [7]:
cholinergic_annotated <- readRDS(paste0(save_dir,'rds_files/cholinergic_annotated.rds'))

In [8]:
# Prepare counts for the cholinergic Seurat object
cholinergic.counts.split <- prepare_counts(cholinergic_annotated, test.category='stage', 
                                           by.category='cholinergic_type', min.nuclei = 50)

In [9]:
# Remove conditions with only one replicate and cell types with only one condition
cholinergic.counts.split <- remove_conditions(cholinergic.counts.split)

In [10]:
# Run DESeq for each cholinergic type
cholinergic_names <- names(cholinergic.counts.split)

if(!dir.exists(paste0(save_dir, 'DESeq2/Cholinergic_Type/'))){
  dir.create(paste0(save_dir, 'DESeq2/Cholinergic_Type/'))
}

path <- paste0(save_dir, 'DESeq2/Cholinergic_Type/')

for (cholinergic_name in cholinergic_names) {
    counts <- cholinergic.counts.split[[cholinergic_name]]
    run_deseq2(category.name = cholinergic_name, counts = counts, savedir.path= path)
}

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting

### Differential expression in non-cluster 2 alpha MNs with disease
Non-c2_Alpha_MNs

In [11]:
alpha_subset <- readRDS(file = paste0(save_dir, 'rds_files/alpha_subset.rds'))

In [12]:
alpha_non_c2 <- subset(alpha_subset, subset = seurat_clusters == 2, invert = TRUE)

In [13]:
# Prepare counts for the Seurat object
non.c2.counts.split <- prepare_counts(alpha_non_c2, test.category='stage', 
                                           by.category='stage', min.nuclei = 50)

In [14]:
counts <- cbind(non.c2.counts.split[[1]], non.c2.counts.split[[2]], non.c2.counts.split[[3]], non.c2.counts.split[[4]])

In [15]:
# Create a data frame of sample information
colData <- data.frame(samples = colnames(counts))

In [16]:
# Add a condition column containing the test.category information (ex: cell class) for each sample
colData$condition <- sub("_.*", "", colData$samples) 

In [17]:
# Convert the sample names to row names
colData <- colData %>% column_to_rownames(var = "samples")

In [18]:
# Create DESEq object
dds <- DESeqDataSetFromMatrix(countData = counts,
                      colData = colData,
                      design = ~ condition)

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”


In [19]:
# Run DESeq
dds <- DESeq(dds, minReplicatesForReplace = Inf)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [20]:
results_names <- resultsNames(dds)
results_names <- results_names[grep("^condition", results_names)]
results_names

[1] "condition_sod.early_vs_ctl" "condition_sod.end_vs_ctl"  
[3] "condition_sod.mid_vs_ctl"

In [21]:
savedir.path <- file.path(save_dir, "DESeq2_rerun", "Non-c2_Alpha_MNs")
dir.create(savedir.path, recursive = TRUE, showWarnings = FALSE)

for (result_name in results_names) {
  res <- results(dds, name = result_name) %>%
    as.data.frame() %>%
    arrange(padj) %>%
    filter(!is.na(padj))
  
  out_file <- file.path(
    savedir.path,
    paste0("Non-c2_Alpha_MNs_", sub("^[^_]*_", "", result_name), ".csv")
  )
  
  write.csv(res, file = out_file)
}

### Differential expression in cluster 2, DAMN alpha MNs vs. other alpha MNs
Other.Clusters_vs_Cluster.2.csv

In [22]:
alpha_subset <- readRDS(file = paste0(save_dir, 'rds_files/alpha_subset.rds'))

In [23]:
alpha_subset$c2 <- ifelse(alpha_subset$seurat_clusters == 2, "Cluster 2", "Other Clusters")

In [24]:
# Prepare counts for the Seurat object
all.counts.split <- prepare_counts(alpha_subset, test.category='c2', 
                                           by.category='c2', min.nuclei = 50)

In [25]:
counts <- cbind(all.counts.split$`Cluster 2`, all.counts.split$`Other Clusters`)

In [26]:
# Create a data frame of sample information
colData <- data.frame(samples = colnames(counts))

In [27]:
# Add a condition column containing the test.category information (ex: cell class) for each sample
colData$condition <- sub("_.*", "", colData$samples) 

In [28]:
# Convert the sample names to row names
colData <- colData %>% column_to_rownames(var = "samples")

In [29]:
# Create DESEq object
dds <- DESeqDataSetFromMatrix(countData = counts,
                      colData = colData,
                      design = ~ condition)

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]



In [30]:
# Run DESeq
dds <- DESeq(dds, minReplicatesForReplace = Inf)

estimating size factors

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This is a message, not a warning or an error]

final dispersion estimates

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names in R. [This 

In [31]:
results_names <- resultsNames(dds)
results_names <- results_names[grep("^condition", results_names)]
results_names

[1] "condition_Other.Clusters_vs_Cluster.2"

In [32]:
res <- results(dds, name = results_names) %>% as.data.frame(.) %>% arrange(padj) %>% filter(!is.na(padj))

In [33]:
write.csv(res, file=paste0(save_dir, paste0('/DESeq2/', sub("^[^_]*_", "", results_names), '.csv')))

### C9, C1 Fast-Firing Alpha MN DESeq2 analysis
C9_C1_FF_Alpha_MNs

In [34]:
# Load alpha MN RNA Seurat object
alpha_subset <- readRDS(file = paste0(save_dir, 'rds_files/alpha_subset.rds'))

In [35]:
# Subset alpha MNs from clusters 0 and 10 (slow-firing)
alpha_c9_c1 <- subset(alpha_subset, seurat_clusters %in% c(9, 1))

In [36]:
# Update orig.ident metadata column to contain stage information and add condition metadata
alpha_c9_c1$orig.ident <- paste0(alpha_c9_c1@meta.data$orig.ident, "_", alpha_c9_c1$stage)
alpha_c9_c1$condition <- ifelse(alpha_c9_c1$stage == "ctl", "control", "sod1")

In [37]:
# Exclude early-stage MNs
alpha_c9_c1 <- subset(alpha_c9_c1, stage != "sod.early")

In [38]:
cell_counts <- alpha_c9_c1@meta.data %>%
    group_by(orig.ident, stage) %>%
    summarize(count = n())

`summarise()` has grouped output by 'orig.ident'. You can override using the `.groups` argument.


In [39]:
# Prepare counts for the Seurat object
alpha.c9.c1.counts.split <- prepare_counts(alpha_c9_c1, test.category='condition', 
                                           by.category='condition', min.nuclei = 30)

In [40]:
counts <- cbind(alpha.c9.c1.counts.split[[1]], alpha.c9.c1.counts.split[[2]])

In [41]:
# Create a data frame of sample information
colData <- data.frame(samples = colnames(counts))

In [42]:
# Add a condition column containing the test.category information (ex: cell class) for each sample
colData$condition <- sub("_.*", "", colData$samples) 

In [43]:
# Convert the sample names to row names
colData <- colData %>% column_to_rownames(var = "samples")

In [44]:
# Create DESEq object
dds <- DESeqDataSetFromMatrix(countData = counts,
                      colData = colData,
                      design = ~ condition)

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”


In [45]:
# Run DESeq
dds <- DESeq(dds, minReplicatesForReplace = Inf)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [46]:
results_names <- resultsNames(dds)
results_names <- results_names[grep("^condition", results_names)]
results_names

[1] "condition_sod1_vs_control"

In [47]:
res <- results(dds, name = results_names) %>% as.data.frame(.) %>% arrange(padj) %>% filter(!is.na(padj))

In [48]:
savedir.path <- paste0(save_dir, 'DESeq2/')
write.csv(res, file=paste0(savedir.path, "C9_C1_FF_Alpha_MNs_", sub("^[^_]*_", "", results_names), '.csv'))

### C0, C10 Slow-Firing Alpha DESeq2 MN analysis
C0_C10_SF_Alpha_MNs

In [49]:
# Load alpha MN RNA Seurat object
alpha_subset <- readRDS(file = paste0(save_dir, 'rds_files/alpha_subset.rds'))

In [50]:
# Subset alpha MNs from clusters 0 and 10 (slow-firing)
alpha_c0_c10 <- subset(alpha_subset, seurat_clusters %in% c(0, 10))

In [51]:
# Update orig.ident metadata column to contain stage information and add condition metadata
alpha_c0_c10$orig.ident <- paste0(alpha_c0_c10@meta.data$orig.ident, "_", alpha_c0_c10$stage)
alpha_c0_c10$condition <- ifelse(alpha_c0_c10$stage == "ctl", "control", "sod1")

In [52]:
# Exclude early-stage MNs
alpha_c0_c10 <- subset(alpha_c0_c10, stage != "sod.early")

In [53]:
cell_counts <- alpha_c0_c10@meta.data %>%
    group_by(orig.ident, stage) %>%
    summarize(count = n())

`summarise()` has grouped output by 'orig.ident'. You can override using the `.groups` argument.


In [54]:
# Prepare counts for the Seurat object
alpha.c0.c10.counts.split <- prepare_counts(alpha_c0_c10, test.category='condition', 
                                           by.category='condition', min.nuclei = 30)

In [55]:
counts <- cbind(alpha.c0.c10.counts.split[[1]], alpha.c0.c10.counts.split[[2]])

In [56]:
# Create a data frame of sample information
colData <- data.frame(samples = colnames(counts))

In [57]:
# Add a condition column containing the test.category information (ex: cell class) for each sample
colData$condition <- sub("_.*", "", colData$samples) 

In [58]:
# Convert the sample names to row names
colData <- colData %>% column_to_rownames(var = "samples")

In [59]:
# Create DESEq object
dds <- DESeqDataSetFromMatrix(countData = counts,
                      colData = colData,
                      design = ~ condition)

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”


In [60]:
# Run DESeq
dds <- DESeq(dds, minReplicatesForReplace = Inf)

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



In [61]:
results_names <- resultsNames(dds)
results_names <- results_names[grep("^condition", results_names)]
results_names

[1] "condition_sod1_vs_control"

In [62]:
res <- results(dds, name = results_names) %>% as.data.frame(.) %>% arrange(padj) %>% filter(!is.na(padj))

In [63]:
savedir.path <- paste0(save_dir, 'DESeq2/')
write.csv(res, file=paste0(savedir.path, "C0_C10_SF_Alpha_MNs_", sub("^[^_]*_", "", results_names), '.csv'))

## Downsampled DESeq2 analyses

### End-stage vs. control
Downsampled_Cholinergic_Type

In [186]:
cholinergic_annotated <- readRDS(paste0(save_dir,'rds_files/cholinergic_annotated.rds'))

In [309]:
# Downsample data so all cholinergic types have the same number of replicates and nuclei per replicate
cholinergic_ctl_end <- cholinergic_annotated
cholinergic_ctl_end <- subset(cholinergic_ctl_end, subset = stage %in% c("ctl", "sod.end"))

In [310]:
# Combine the orig.ident and stage columns with an underscore, and store as a new metadata column
cholinergic_ctl_end@meta.data$orig.ident_stage <- paste(cholinergic_ctl_end@meta.data$orig.ident, cholinergic_ctl_end@meta.data$stage, sep = "_")

In [311]:
# Calculate the number of nuclei for each orig.ident_stage within each cholinergic type
nuclei_counts <- table(cholinergic_ctl_end@meta.data$orig.ident_stage, cholinergic_ctl_end@meta.data$cholinergic_type)
nuclei_counts_df <- as.data.frame(nuclei_counts)
colnames(nuclei_counts_df) <- c("orig.ident_stage", "cholinergic_type", "nuclei_count")

# Filter out rows with nuclei_count < 25
nuclei_counts_df <- dplyr::filter(nuclei_counts_df, nuclei_count >= 25)

In [312]:
# Get unique orig.ident_stage values for each cholinergic type
orig_ident_by_type <- nuclei_counts_df %>%
  group_by(cholinergic_type) %>%
  summarize(orig_ident = list(unique(orig.ident_stage))) %>%
  ungroup()

# Find the intersection of orig.ident_stage values across all cholinergic types
shared_orig_ident <- Reduce(intersect, orig_ident_by_type$orig_ident)

In [314]:
# Filter out rows from nuclei_counts_df where orig.ident_stage is not in shared_orig_ident
nuclei_counts_shared <- dplyr::filter(nuclei_counts_df, orig.ident_stage %in% shared_orig_ident)

In [315]:
# Create the dataframe with smallest_nuclei_count
smallest_counts <- nuclei_counts_shared %>%
  group_by(orig.ident_stage) %>%
  summarize(smallest_nuclei_count = min(nuclei_count))

In [316]:
# Keep data from orig.ident_stage values that are shared across all cholinergic types
cholinergic_ctl_end <- subset(cholinergic_ctl_end, subset = orig.ident_stage %in% shared_orig_ident)

# Combine the cholinergic_type and orig.ident_stage columns with an underscore, and store as a new metadata column
cholinergic_ctl_end@meta.data$type_orig.ident_stage <- paste(cholinergic_ctl_end@meta.data$cholinergic_type, cholinergic_ctl_end@meta.data$orig.ident_stage, sep = "_")

In [317]:
# Split the Seurat object by type_orig.ident_stage
cholinergic_list <- SplitObject(cholinergic_ctl_end, split.by = "type_orig.ident_stage")

In [318]:
# Define the downsampling function
downsample_function <- function(seurat_obj, smallest_counts_df) {
  # Get the orig.ident_stage value from the Seurat object
  orig_ident_stage <- seurat_obj@meta.data$orig.ident_stage[1]
  
  # Get the smallest nuclei count for the orig.ident_stage value
  smallest_count <- smallest_counts_df$smallest_nuclei_count[smallest_counts_df$orig.ident_stage == orig_ident_stage]    

  # Apply downsampling
  downsampled_data <- seurat_obj[, sample(colnames(seurat_obj), size = smallest_count, replace = FALSE)]
  return(downsampled_data)
}

In [319]:
# Apply downsampling function to each Seurat object in seurat_list
downsampled_cholinergic_list <- lapply(cholinergic_list, downsample_function, smallest_counts_df = smallest_counts)

In [320]:
# Merge the downsampled Seurat objects
downsampled_cholinergic_obj <- merge(downsampled_cholinergic_list[[1]], downsampled_cholinergic_list[-1], merge.data = TRUE)

In [321]:
downsampled_cholinergic_obj

An object of class Seurat 
64088 features across 12684 samples within 3 assays 
Active assay: RNA (31044 features, 0 variable features)
 2 other assays present: original_counts, integrated

In [324]:
# Prepare counts for the cholinergic Seurat object
cholinergic.counts.split <- prepare_counts(downsampled_cholinergic_obj, test.category='stage', 
                                           by.category='cholinergic_type', min.nuclei = 25)

In [325]:
# Run DESeq for each cholinergic type
cholinergic_names <- names(cholinergic.counts.split)

if(!dir.exists(paste0(save_dir, 'DESeq2/Downsampled_Cholinergic_Type/'))){
  dir.create(paste0(save_dir, 'DESeq2/Downsampled_Cholinergic_Type/'))
}

path <- paste0(save_dir, 'DESeq2/Downsampled_Cholinergic_Type/')

for (cholinergic_name in cholinergic_names) {
    counts <- cholinergic.counts.split[[cholinergic_name]]
    run_deseq2(category.name = cholinergic_name, counts = counts, savedir.path= path)
}

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting

### Mid-stage vs. control
Downsampled_Cholinergic_Type_Mid

In [36]:
cholinergic_annotated <- readRDS(paste0(save_dir,'rds_files/cholinergic_annotated.rds'))

In [36]:
# Downsample data so all cholinergic types have the same number of replicates and nuclei per replicate
cholinergic_ctl_mid <- cholinergic_annotated
cholinergic_ctl_mid <- subset(cholinergic_ctl_mid, subset = stage %in% c("ctl", "sod.mid"))

In [37]:
# Remove cholinergic interneurons 
cholinergic_ctl_mid <- subset(cholinergic_ctl_mid, subset = cholinergic_type %in% c("Alpha MNs", "Gamma MNs", "Gamma Star MNs", "Visceral Sacral MNs", "Visceral Thoracic MNs", "Visceral Thoracic MNs"))

In [38]:
# Combine the orig.ident and stage columns with an underscore, and store as a new metadata column
cholinergic_ctl_mid@meta.data$orig.ident_stage <- paste(cholinergic_ctl_mid@meta.data$orig.ident, cholinergic_ctl_mid@meta.data$stage, sep = "_")

In [39]:
# Calculate the number of nuclei for each orig.ident_stage within each cholinergic type
nuclei_counts <- table(cholinergic_ctl_mid@meta.data$orig.ident_stage, cholinergic_ctl_mid@meta.data$cholinergic_type)
nuclei_counts_df <- as.data.frame(nuclei_counts)
colnames(nuclei_counts_df) <- c("orig.ident_stage", "cholinergic_type", "nuclei_count")

# Filter out rows with nuclei_count < 25
nuclei_counts_df <- dplyr::filter(nuclei_counts_df, nuclei_count >= 25)

In [40]:
# Get unique orig.ident_stage values for each cholinergic type
orig_ident_by_type <- nuclei_counts_df %>%
  group_by(cholinergic_type) %>%
  summarize(orig_ident = list(unique(orig.ident_stage))) %>%
  ungroup()

# Find the intersection of orig.ident_stage values across all cholinergic types
shared_orig_ident <- Reduce(intersect, orig_ident_by_type$orig_ident)

In [42]:
# Filter out rows from nuclei_counts_df where orig.ident_stage is not in shared_orig_ident
nuclei_counts_shared <- dplyr::filter(nuclei_counts_df, orig.ident_stage %in% shared_orig_ident)

In [43]:
# Create the dataframe with smallest_nuclei_count
smallest_counts <- nuclei_counts_shared %>%
  group_by(orig.ident_stage) %>%
  summarize(smallest_nuclei_count = min(nuclei_count))

In [44]:
# Keep data from orig.ident_stage values that are shared across all cholinergic types
cholinergic_ctl_mid <- subset(cholinergic_ctl_mid, subset = orig.ident_stage %in% shared_orig_ident)

# Combine the cholinergic_type and orig.ident_stage columns with an underscore, and store as a new metadata column
cholinergic_ctl_mid@meta.data$type_orig.ident_stage <- paste(cholinergic_ctl_mid@meta.data$cholinergic_type, cholinergic_ctl_mid@meta.data$orig.ident_stage, sep = "_")

In [45]:
# Split the Seurat object by type_orig.ident_stage
cholinergic_list <- SplitObject(cholinergic_ctl_mid, split.by = "type_orig.ident_stage")

In [46]:
# Define the downsampling function
downsample_function <- function(seurat_obj, smallest_counts_df) {
  # Get the orig.ident_stage value from the Seurat object
  orig_ident_stage <- seurat_obj@meta.data$orig.ident_stage[1]
  
  # Get the smallest nuclei count for the orig.ident_stage value
  smallest_count <- smallest_counts_df$smallest_nuclei_count[smallest_counts_df$orig.ident_stage == orig_ident_stage]    

  # Apply downsampling
  downsampled_data <- seurat_obj[, sample(colnames(seurat_obj), size = smallest_count, replace = FALSE)]
  return(downsampled_data)
}

In [47]:
# Apply downsampling function to each Seurat object in seurat_list
downsampled_cholinergic_list <- lapply(cholinergic_list, downsample_function, smallest_counts_df = smallest_counts)

In [48]:
# Merge the downsampled Seurat objects
downsampled_cholinergic_obj <- merge(downsampled_cholinergic_list[[1]], downsampled_cholinergic_list[-1], merge.data = TRUE)

In [49]:
# Prepare counts for the cholinergic Seurat object
cholinergic.counts.split <- prepare_counts(downsampled_cholinergic_obj, test.category='stage', 
                                           by.category='cholinergic_type', min.nuclei = 25)

In [50]:
# Run DESeq for each cholinergic type
cholinergic_names <- names(cholinergic.counts.split)

if(!dir.exists(paste0(save_dir, 'DESeq2/Downsampled_Cholinergic_Type_Mid/'))){
  dir.create(paste0(save_dir, 'DESeq2/Downsampled_Cholinergic_Type_Mid/'))
}

path <- paste0(save_dir, 'DESeq2/Downsampled_Cholinergic_Type_Mid/')

for (cholinergic_name in cholinergic_names) {
    counts <- cholinergic.counts.split[[cholinergic_name]]
    run_deseq2(category.name = cholinergic_name, counts = counts, savedir.path= path)
}

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting

### Early-stage vs. control
Downsampled_Cholinergic_Type_early

In [9]:
cholinergic_annotated <- readRDS(paste0(save_dir,'rds_files/cholinergic_annotated.rds'))

In [51]:
# Downsample data so all cholinergic types have the same number of replicates and nuclei per replicate
cholinergic_ctl_early <- cholinergic_annotated
cholinergic_ctl_early <- subset(cholinergic_ctl_early, subset = stage %in% c("ctl", "sod.early"))

In [52]:
# Remove cholinergic interneurons 
cholinergic_ctl_early <- subset(cholinergic_ctl_early, subset = cholinergic_type %in% c("Alpha MNs", "Gamma MNs", "Gamma Star MNs", "Visceral Sacral MNs", "Visceral Thoracic MNs"))

In [53]:
# Combine the orig.ident and stage columns with an underscore, and store as a new metadata column
cholinergic_ctl_early@meta.data$orig.ident_stage <- paste(cholinergic_ctl_early@meta.data$orig.ident, cholinergic_ctl_early@meta.data$stage, sep = "_")

In [54]:
# Calculate the number of nuclei for each orig.ident_stage within each cholinergic type
nuclei_counts <- table(cholinergic_ctl_early@meta.data$orig.ident_stage, cholinergic_ctl_early@meta.data$cholinergic_type)
nuclei_counts_df <- as.data.frame(nuclei_counts)
colnames(nuclei_counts_df) <- c("orig.ident_stage", "cholinergic_type", "nuclei_count")

# Filter out rows with nuclei_count < 25
nuclei_counts_df <- dplyr::filter(nuclei_counts_df, nuclei_count >= 25)

In [56]:
# Get unique orig.ident_stage values for each cholinergic type
orig_ident_by_type <- nuclei_counts_df %>%
  group_by(cholinergic_type) %>%
  summarize(orig_ident = list(unique(orig.ident_stage))) %>%
  ungroup()

# Find the intersection of orig.ident_stage values across all cholinergic types
shared_orig_ident <- Reduce(intersect, orig_ident_by_type$orig_ident)

In [58]:
# Filter out rows from nuclei_counts_df where orig.ident_stage is not in shared_orig_ident
nuclei_counts_shared <- dplyr::filter(nuclei_counts_df, orig.ident_stage %in% shared_orig_ident)

In [59]:
# Create the dataframe with smallest_nuclei_count
smallest_counts <- nuclei_counts_shared %>%
  group_by(orig.ident_stage) %>%
  summarize(smallest_nuclei_count = min(nuclei_count))

In [60]:
# Keep data from orig.ident_stage values that are shared across all cholinergic types
cholinergic_ctl_early <- subset(cholinergic_ctl_early, subset = orig.ident_stage %in% shared_orig_ident)

# Combine the cholinergic_type and orig.ident_stage columns with an underscore, and store as a new metadata column
cholinergic_ctl_early@meta.data$type_orig.ident_stage <- paste(cholinergic_ctl_early@meta.data$cholinergic_type, cholinergic_ctl_early@meta.data$orig.ident_stage, sep = "_")

In [61]:
# Split the Seurat object by type_orig.ident_stage
cholinergic_list <- SplitObject(cholinergic_ctl_early, split.by = "type_orig.ident_stage")

In [62]:
# Define the downsampling function
downsample_function <- function(seurat_obj, smallest_counts_df) {
  # Get the orig.ident_stage value from the Seurat object
  orig_ident_stage <- seurat_obj@meta.data$orig.ident_stage[1]
  
  # Get the smallest nuclei count for the orig.ident_stage value
  smallest_count <- smallest_counts_df$smallest_nuclei_count[smallest_counts_df$orig.ident_stage == orig_ident_stage]    

  # Apply downsampling
  downsampled_data <- seurat_obj[, sample(colnames(seurat_obj), size = smallest_count, replace = FALSE)]
  return(downsampled_data)
}

In [63]:
# Apply downsampling function to each Seurat object in seurat_list
downsampled_cholinergic_list <- lapply(cholinergic_list, downsample_function, smallest_counts_df = smallest_counts)

In [64]:
# Merge the downsampled Seurat objects
downsampled_cholinergic_obj <- merge(downsampled_cholinergic_list[[1]], downsampled_cholinergic_list[-1], merge.data = TRUE)

In [65]:
# Prepare counts for the cholinergic Seurat object
cholinergic.counts.split <- prepare_counts(downsampled_cholinergic_obj, test.category='stage', 
                                           by.category='cholinergic_type', min.nuclei = 25)

In [66]:
# Run DESeq for each cholinergic type
cholinergic_names <- names(cholinergic.counts.split)

if(!dir.exists(paste0(save_dir, 'DESeq2/Downsampled_Cholinergic_Type_early/'))){
  dir.create(paste0(save_dir, 'DESeq2/Downsampled_Cholinergic_Type_early/'))
}

path <- paste0(save_dir, 'DESeq2/Downsampled_Cholinergic_Type_early/')

for (cholinergic_name in cholinergic_names) {
    counts <- cholinergic.counts.split[[cholinergic_name]]
    run_deseq2(category.name = cholinergic_name, counts = counts, savedir.path= path)
}

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting